# Tiny GPT

U ovom primeru napravićemo minijaturni GPT koji radi na niovu karaktera. 

Ceo program se oslanja na tri glavne ideje:

1. **Karakteri postaju tokeni sa jedinstvenim identifikatorom.** Neuronske mreže rade nad brojevima, pa svaki karakter u korpusu dobija stabilan celobrojni ID.
2. **Trening primeri su tekstualni prozori.** Ulaz je kratka sekvenca, a izlaz sledeći karakter iz sekvence na prozoru iste dužine pomeren pomerena za jedan karakter unapred.
3. **Kauzalna pažnja blokira budućnost.** Na poziciji `t`, model može da koristi karaktere na pozicijama `0..t`, ali ne i karaktere posle pozicije `t`.

Model je sličan GPT-u zato što je dekoderski, kauzalan, autoregresivan i predviđa sledeći token na osnovu prethodnih tokena. Mali je zato što su korpus, širina modela, broj slojeva i dužina konteksta namerno mali. Model neće biti neki sjajan pisac, ali principi obučavanja su isti kao kod većih kauzalnih jezičkih modela.

U suštini, model daje odgovor na pitanje: **koji karakter dolazi sledeći, na osnovu već viđenih karaktera?**

1. Napraviti mali rečnik iz teksta.
2. Pretvoriti ceo tekst u celobrojne token ID-jeve.
3. Nasumično uzorkovati ulazne i izlazne batch-eve iz token ID-jeva.
4. Napraviti sloj kauzalne samopažnje.
5. Povezati slojeve pažnje i propagacije u napred u mali *Transformer*.
6. Obučiti model da predviđa sledeći karakter.
7. Generisati tekst uzastopnim uzorkovanjem sledećeg karaktera.


In [40]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import tensorflow as tf
import numpy as np
from matplotlib import pyplot as plt

## Mali korpus i rečnik

Niska `text` u narednoj ćeliji je celo korpus (literatura) za ovaj mali model. 
Model ne zna engleski jezik. Vidi samo nisku `text`, pa svaki obrazac koji nauči mora da potiče iz karaktera koji su toj nisci nalaze.

Pošto je tekst unutar trostrukih navodnika, uključuje i znakove za novi red. Ti znakovi su pravi trening podaci, isto kao `h`, `e` ili razmak.

- `stoi` znači "string to integer" i mapira svaki karakter u njegov token ID.
- `itos` znači "integer to string" i mapira svaki token ID nazad u karakter.

Primer:

```python
chars = ['a', 'b', 'n']
stoi = {'a': 0, 'b': 1, 'n': 2}
itos = {0: 'a', 1: 'b', 2: 'n'}
```

Model ne može da trenira nad sirovim karakterima, pa `encode` pretvara karakter (nisku) u token ID-jeve. `decode` pretvara token ID u karakter iz niske. Ako je `stoi = {'a': 0, 'b': 1, 'n': 2}`, onda `encode("banana")` vraća:

```python
np.array([1, 0, 2, 0, 2, 0], dtype=np.int32)
```

Na kraju ćelije, `data = encode(text)` pretvara ceo korpus u jednodimenzionalni niz celobrojnih token ID-jeva.


In [41]:
text = """
hello world this is a tiny dataset for a minimal gpt model
we will train a character level transformer on this text only
"""

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for c,i in stoi.items()}

def encode(s): return np.array([stoi[c] for c in s], dtype=np.int32)
def decode(l): return ''.join([itos[i] for i in l])

data = encode(text)

## Generisanje batch-eva

Generator batch-eva pravi zadatke za vežbanje modela.

`block_size = 32` je dužina konteksta. Svaki trening primer sadrži 32 ulazna karaktera. `batch_size = 16` je broj primera koji se obrađuju zajedno, pa svaki ulazni batch ima oblik:

```text
(batch_size, block_size) = (16, 32)
```

`get_batch()` bira nasumične početne pozicije u kodiranom korpusu. Svaki ceo broj je početak jednog prozora za trening. Gornja granica je konzervativna zato što `np.random.randint` ne uključuje krajnju vrednost, pa se poslednji mogući prozor ne koristi. To ovde ne smeta jer je korpus mali, a cilj je da se prikaže opšti princip obučavanja jezičkog modela.

Ulazni batch `x` sadrži trenutne karaktere. Ciljni batch `y` sadrži sledeće karaktere. Svaki cilj je pomeren za jednu poziciju udesno.

Primer sa `block_size = 5`:

```text
tekstualni prozor: hello world

x: h e l l o
y: e l l o _
```

Donja crta predstavlja razmak. Prvi ulazni karakter `h` traži od modela da predvidi `e`. Sekvenca `h e` traži od modela da predvidi `l`. Sekvenca `h e l l` traži od modela da predvidi `o`.

I upravo je glavna ideja modela da na svakoj poziciji predvidi sledeći karakter na osnovu prethodnih.

Oba vraćena niza imaju oblik `(16, 32)`.

Funkcija `dataset()` je generator batch-eva u za TensorFlow. Svaki put kada TensorFlow zatraži podatke, napravi se novi nasumični batch. `output_signature` govori TensorFlow-u oblik i tip podataka za `x` i `y`, a `prefetch(tf.data.AUTOTUNE)` omogućava TensorFlow-u da priprema buduće batch-eve dok model trenira nad trenutnim batch-em.


In [42]:
block_size = 32
batch_size = 16

def get_batch():
    ix = np.random.randint(0, len(data) - block_size - 1, size=batch_size)
    x = np.stack([data[i:i+block_size] for i in ix])
    y = np.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

def dataset():
    def gen():
        while True:
            x, y = get_batch()
            yield x, y

    return tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(batch_size, block_size), dtype=tf.int32),
            tf.TensorSpec(shape=(batch_size, block_size), dtype=tf.int32),
        )
    ).prefetch(tf.data.AUTOTUNE)

## Kauzalna samopažnja

Kauzalna samopažnja omogućava svakoj poziciji da postavi pitanje: "Na koje prethodne karaktere treba da obratim pažnju kada predviđam sledeći?"

`d_model` je širina embeddinga: broj karakteristika kojima se predstavlja svaka pozicija tokena. 
`n_heads` je broj glava pažnje. Više glava omogućava modelu da istovremeno traži različite vrste odnosa. 
`d_model` bi trebalo da bude ravnomerno podeljen na glave. Kada je `d_model = 64`, a `n_heads = 4`, svaka glava dobija `head_dim = 16` karakteristika.

Sloj pravi dve guste transformacije:

- `self.qkv` pravi tri vektora za svaku poziciju tokena: upit (eng. **upit**), ključ (eng. key) i vrednost (eng. value).
- `self.proj` šalje izlaz sloja *pažnje* nazad u `d_model` karakteristika.

Najjednostavnije: zamislite da svaki karakter u sekvenci dobije tri kartice.

- **Upit** kartica kaže: "Šta mi treba?"
- **Ključ** kartica kaže: "Šta ja nudim?"
- **Vrednost** kartica kaže: "Koju informaciju ću dati ako me neko izabere?"

Kada model obrađuje jednu poziciju, njen **upit** se poredi sa **ključevima** svih dozvoljenih pozicija. Ako se **upit** pozicije `i` dobro poklopi sa **ključem** pozicije `j`, onda pozicija `i` obraća više pažnje na poziciju `j`. Posle softmax-a, ta pažnja postaje težina. Te težine zatim određuju koliko vektor **value** svake pozicije ulazi u novi vektor za poziciju `i`.

Primer: ako model gleda poslednje slovo u `hello`, **upit** tog slova može da traži "šta mi pomaže da predvidim sledeći karakter?" ključevi ranijih slova govore šta ta slova mogu da ponude. Vektori **vrednosti** nose stvarnu informaciju koja se meša u rezultat.

Ime `qkv` znači da jedna `Dense` transformacija izračuna sva tri vektora odjednom. Za svaki token ona napravi jedan vektor dužine `3 * d_model`, u kojem su **upit**, **ključ** i **vrednost** nadovezani jedan na drugi:

```text
qkv = [upit | ključ | vrednost]
```

Zatim `tf.split(qkv, 3, axis=-1)` razdvoji taj vektor na tri posebna tenzora: `q`, `k` i `v`. To je efikasnije nego da se prave tri odvojena sloja `Dense`, ali ideja je ista.

Metod `call` prima tenzor oblika:

```text
x: (B, T, C)
```

gde je `B` veličina batch-a, `T` dužina sekvence, a `C` širina kanala, obično jednaka `d_model`. 
Tokom treninga u ovoj svesci očekivane dimenzije su:

```text
B = 16
T = 32
C = 64
```

Prvi gusti sloj proizvodi sva tri tenzora odjednom:

```text
qkv: (B, T, 3 * d_model)
q:   (B, T, d_model)
k:   (B, T, d_model)
v:   (B, T, d_model)
```

Svaki tenzor se zatim preoblikuje i transponuje kako bi pažnja mogla da se računa odvojeno po glavama:

```text
q, k, v posle transponovanja: (B, n_heads, T, head_dim)
```

Skorovi pažnje porede **upite** sa **ključevima**. Množenje matrica poredi svaku poziciju sa svakom drugom pozicijom:

```text
q:   (B, n_heads, T, head_dim)
k^T: (B, n_heads, head_dim, T)
att: (B, n_heads, T, T)
```

Za svaki red, `att[..., i, j]` pita koliko pozicija `i` treba da obrati pažnju na poziciju `j`. Deljenje sa `sqrt(head_dim)` sprečava da skalarni proizvodi postanu preveliki, što sprečava da softmax prerano postane previše oštra tokom treninga.

Kauzalna maska sprečava curenje budućih informacija unazad. Ona je donje trougaona:

```text
1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1
```

`1` znači da je pažnja dozvoljena. `0` znači da je pažnja blokirana. Pozicija 0 može da vidi samo poziciju 0. Pozicija 1 može da vidi pozicije 0 i 1. Pozicija 2 može da vidi pozicije 0, 1 i 2.

Blokirane pozicije dobijaju veoma velike negativne brojeve. Posle softmax-a, verovatnoće na tim pozicijama biće veoma blizu nule. Svaki red preko poslednje dimenzije tada postaje raspodela od dozvoljenih prethodnih pozicija.

Težine pažnje biraju mešavinu vektora **vrednosti**:

```text
att: (B, n_heads, T, T)
v:   (B, n_heads, T, head_dim)
out: (B, n_heads, T, head_dim)
```

Glave se vraćaju u prvobitni raspored, ponovo spajaju u `(B, T, C)` i projektuju nazad u `(B, T, d_model)`. Sloj pažnje počinje sa jednim vektorom po poziciji tokena i vraća jedan poboljšani vektor po poziciji tokena.


In [43]:
class CausalSelfAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = tf.keras.layers.Dense(3 * d_model)
        self.proj = tf.keras.layers.Dense(d_model)

    def call(self, x):
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]
        C = x.shape[-1]

        qkv = self.qkv(x)
        q, k, v = tf.split(qkv, 3, axis=-1)

        q = tf.reshape(q, (B, T, self.n_heads, self.head_dim))
        k = tf.reshape(k, (B, T, self.n_heads, self.head_dim))
        v = tf.reshape(v, (B, T, self.n_heads, self.head_dim))

        q = tf.transpose(q, [0, 2, 1, 3])
        k = tf.transpose(k, [0, 2, 1, 3])
        v = tf.transpose(v, [0, 2, 1, 3])

        att = tf.matmul(q, k, transpose_b=True) / tf.math.sqrt(tf.cast(self.head_dim, tf.float32))

        mask = tf.linalg.band_part(tf.ones((T, T)), -1, 0)
        att = att * mask + (1.0 - mask) * (-1e10)

        att = tf.nn.softmax(att, axis=-1)
        out = tf.matmul(att, v)

        out = tf.transpose(out, [0, 2, 1, 3])
        out = tf.reshape(out, (B, T, C))
        return self.proj(out)

## Blok `Transformer`

Blok `Transformer` poboljšava reprezentacije tokena pomoću dva podsloja: kauzalne pažnje i propagacije unapred.

Podsloj pažnje omogućava pozicijama da razmenjuju informacije sa prethodnim pozicijama. Sloj propagacije u napred obrađuje svaku poziciju nezavisno nakon što je pažnja izmešala kontekstualne informacije.

Oblik se čuva:

```text
ulaz:  (B, T, d_model)
izlaz: (B, T, d_model)
```


In [44]:
class Block(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.att = CausalSelfAttention(d_model, n_heads)
        self.ff = tf.keras.Sequential([
            tf.keras.layers.Dense(4 * d_model, activation="relu"),
            tf.keras.layers.Dense(d_model)
        ])
        self.ln1 = tf.keras.layers.LayerNormalization()
        self.ln2 = tf.keras.layers.LayerNormalization()

    def call(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

## TinyGPT model

`TinyGPT` sastavlja kompletan jezički model na nivou karaktera.

Podrazumevani hiperparametri:

```text
vocab_size = broj jedinstvenih karaktera
d_model = 64
n_heads = 4
n_layers = 2
block_size = 32
```

Ovakav model je dovoljno mali da se brzo trenira i ilustruje minijaturnu arhitekturu sličnih modela.

Token embedding mapira svaki ID karaktera u naučeni vektor dužine `64`. Ako ulaz ima oblik `(B, T)`, token embedding ima oblik `(B, T, d_model)`. Celobrojni ID je samo oznaka; embedding daje tom karakteru naučivu vektorsku reprezentaciju.

Transformeru su potrebne i informacije o poziciji zato što sama pažnja ne zna da li je token došao prvi, drugi ili deseti. Pozicioni embedding daje svakom vremenskom koraku njegov naučeni vektor. 

`tok` ima oblik `(B, T, d_model)`, dok `pos` ima oblik `(T, d_model)`. 
TensorFlow emituje `pos` preko batch dimenzije, pa svaki primer dobija iste pozicione informacije.

`self.blocks` podrazumevano sadrži dva Transformer bloka. `self.ln` normalizuje završna skrivena stanja. `self.head` mapira svaki skriveni vektor u `vocab_size` izlaznih ocena.

Te izlazne ocene se zovu logiti. Logit nije verovatnoća već nenormalizovana ocena koji mogu da koriste unakrsna entropija i softmax.

Model prima token ID-jeve oblika `(B, T)` i vraća logite oblika:

```text
(B, T, vocab_size)
```


In [45]:
class TinyGPT(tf.keras.Model):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, block_size=32):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pos_emb = tf.keras.layers.Embedding(block_size, d_model)
        self.blocks = [Block(d_model, n_heads) for _ in range(n_layers)]
        self.ln = tf.keras.layers.LayerNormalization()
        self.head = tf.keras.layers.Dense(vocab_size)

    def call(self, x):
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]

        tok = self.tok_emb(x)
        pos = self.pos_emb(tf.range(T))
        x = tok + pos

        for blk in self.blocks:
            x = blk(x)

        x = self.ln(x)
        return self.head(x)

## Funkcija greške i obučavanje

Obučavanje uči model da ispravnom sledećem karakteru dodeli visoke ocene.

Model predviđa celobrojne ID-jeve karaktera, pa je odgovarajuća funkcija greška retka kategorička unakrsna entropija. 

Svaki korak u obučavanju modela radi sledeće:

1. Uzima ulazni batch `x` i ciljni batch `y`.
2. Prosleđuje `x` kroz model da dobije logite.
3. Poredi logite sa `y` pomoću unakrsne entropije.
4. Računa gradijente.
5. Ažurira težine modela pomoću Adam optimizatora.

Pošto je korpus mali, model može brzo da nauči obrasce iz teksta.


In [46]:
model = TinyGPT(vocab_size)
optimizer = tf.keras.optimizers.Adam(3e-4)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss=loss_fn
)

history = model.fit(
    dataset(),
    steps_per_epoch=2000,
    epochs=1
)

2000/2000 ━━━━━━━━━━━━━━━━━━━━ 17s 7ms/step - loss: 0.1906


## Generisanje

Generisanje koristi istrenirani model autoregresivno: predvidi jedan karakter, dodaj ga, pa predvidi sledeći karakter iz ažuriranog teksta.

`start` je prompt. Ako je `start = "hello"`, onda `encode(start)` pretvara prompt u token ID-jeve.

Softmax pretvara logite u raspodelu verovatnoće. Na primer:

```text
razmak: 0.70
w:      0.20
t:      0.05
ostalo: 0.05
```

`np.random.choice(vocab_size, p=probs[0])` uzorkuje jedan token ID iz te raspodele. Uzorkovanje se razlikuje od stalnog biranja najveće verovatnoće. Može da proizvede raznovrsnost, ali može da proizvede i čudan tekst, naročito kada je model mali ili slabo istreniran.

Uzorkovani ID se preoblikuje u `(1, 1)` kako bi mogao da se nadoveže na trenutnu sekvencu. Posle jedne iteracije, `idx` ima oblik `(1, prompt_length + 1)`. Posle 200 iteracija, ima oblik `(1, prompt_length + 200)`. Na kraju, `decode` pretvara generisane token ID-jeve nazad u karaktere.


In [47]:
def generate(model, start, max_new=200):
    idx = tf.convert_to_tensor([encode(start)], dtype=tf.int32)

    # Petlja se izvršava jednom za svaki novi karakter. Ako je `max_new = 200`, funkcija uzorkuje 200 dodatnih karaktera.
    for _ in range(max_new):
        
        # Model prihvata najviše `block_size` pozicija, pa se kontekst skraćuje pomoću `idx[:, -block_size:]`. 
        # Time se zadržava poslednjih 32 token ID-ja.
        idx_cond = idx[:, -block_size:]

        # vraća logite za svaku poziciju u trenutnom kontekstu
        logits = model.predict(idx_cond, verbose=0)

        # Samo poslednja pozicija je bitna za sledeći generisani karakter, pa zadržavamo `logits[:, -1, :]`, 
        # koji ima oblik `(1, vocab_size)`. 
        # logits je ocena modela za svaki mogući sledeći karakter posle celog trenutnog prompta.
        logits = logits[:, -1, :]

        # Softmax pretvara logite u raspodelu verovatnoće
        probs = tf.nn.softmax(logits).numpy()

        # Uzorkuje jedan token ID iz te raspodele
        next_id = np.random.choice(vocab_size, p=probs[0])

        # Uskladi dimenzije radi nadovezivanja na trenutnu sekvencu
        next_id = tf.constant([[next_id]], dtype=tf.int32)

        # Nadoveži na trenutnu sekvencu
        idx = tf.concat([idx, next_id], axis=1)

    return decode(idx.numpy()[0])

print(generate(model, "hello"))

hello world this is a tiny dataset for a minimal gpt model
we will train a character level transformer on this text only d timis a tevelor t ptrany danaset  ormsfor  teinima xt odet mini
al gpt model
we wi


## Primer prolaska: jedan trening prozor

Pretpostavimo da korpus počinje ovako:

```text
hello world
```

i da je `block_size = 5`.

Jedan mogući trening primer je:

```text
x tekst: hello
y tekst: ello_
```

Donja crta predstavlja razmak. Model vidi:

```text
h e l l o
```

Trenira se da predvidi:

```text
e l l o _
```

Tako model dobija nekoliko urozoraka:

| Pozicija | Vidljiv ulaz zbog kauzalne maske | Ciljni sledeći karakter |
| --- | --- | --- |
| 0 | `h` | `e` |
| 1 | `h e` | `l` |
| 2 | `h e l` | `l` |
| 3 | `h e l l` | `o` |
| 4 | `h e l l o` | razmak |

Ova tabela je osnova kauzalnog jezičkog modelovanja. Od modela se nikada ne traži da predvidi karakter koristeći buduće karaktere.

## Primer prolaska: jedan korak generisanja

Pretpostavimo da je prompt:

```text
hello
```

Funkcija ga kodira:

```text
[id_h, id_e, id_l, id_l, id_o]
```

Model proizvodi logite za svaku poziciju prompta.

Ako softmax izračuna raspodelu verovatnoće:

```text
razmak: 0.70
w:      0.20
t:      0.05
ostalo: 0.05
```

a uzorkovani karakter je razmak, sekvenca postaje:

```text
hello_
```

Zatim se petlja ponavlja. Model sada predviđa karakter posle `"hello "`, itd..


## Važni oblici tenzora

Ovi oblici su kostur programa.

| Objekat | Oblik | Značenje |
| --- | --- | --- |
| `data` | `(num_characters,)` | Ceo korpus kao token ID-jevi |
| `x` | `(batch_size, block_size)` | Ulazni token ID-jevi |
| `y` | `(batch_size, block_size)` | Ciljni ID-jevi sledećih karaktera |
| `tok` | `(B, T, d_model)` | Token embedding |
| `pos` | `(T, d_model)` | Pozicioni embedding |
| `q`, `k`, `v` pre glava | `(B, T, d_model)` | query, key i value vektori |
| `q`, `k`, `v` posle transponovanja | `(B, n_heads, T, head_dim)` | Reprezentacije po glavama |
| `att` | `(B, n_heads, T, T)` | Skorovi ili težine pažnje |
| izlaz Transformer bloka | `(B, T, d_model)` | Kontekstualni token vektori |
| logiti modela | `(B, T, vocab_size)` | Skorovi za sledeći karakter |
| `idx` tokom generisanja | `(1, current_length)` | Prompt plus uzorkovani tokeni |

Ako je neki od ovih oblika pogrešan, model obično brzo prijavi grešku.


## Rezime

Ova sveska je mala, ali sadrži osnovni tok kauzalnog jezičkog modela:

```text
tekst -> token ID-jevi -> pomereni batch-evi -> kauzalni Transformer -> logiti -> unakrsna entropija -> uzorkovani sledeći tokeni
```
